In [81]:
import pandas as pd

In [82]:
CPchem = pd.read_csv("chemical_annotations.csv")
CPchem

,BROAD_ID,CPD_NAME,CPD_NAME_TYPE,CPD_SAMPLE_ID,DOS_LIBRARY,SOURCE_NAME,CHEMIST_NAME,VENDOR_CATALOG_ID,CPD_SMILES,USERCOMMENT
0,BRD-A56675431-001-04-0,altizide,INN,SA82748,NaN,Prestwick Chemical Inc.,NaN,Prestw-721,NS(=O)(=O)c1cc2c(NC(CSCC=C)NS2(=O)=O)cc1Cl,NaN
1,BRD-A51829654-001-01-4,"BRL-15,572",common,SA82481,NaN,Biomol International Inc.,NaN,AC-536,OC(CN1CCN(CC1)c1cccc(Cl)c1)C(c1ccccc1)c1ccccc1,NaN
2,BRD-K04046242-001-03-6,equilin,primary-common,SA82922,NaN,Prestwick Chemical Inc.,NaN,Prestw-850,C[C@]12CC[C@H]3C(=CCc4cc(O)ccc34)[C@@H]1CCC2=O,NaN
3,BRD-K16508793-001-01-8,diazepam,INN,SA59660,NaN,MicroSource Discovery Systems Inc.,NaN,1900003,CN1c2ccc(Cl)cc2C(=NCC1=O)c1ccccc1,NaN
4,BRD-K09397065-001-01-6,SR 57227A,to-be-curated,SA82504,NaN,Biomol International Inc.,NaN,AC-561,NC1CCN(CC1)c1cccc(Cl)n1,NaN
...,...,...,...,...,...,...,...,...,...,...
30611,BRD-K47092271-001-01-7,BRD-K47092271,BROAD_CPD_ID,SA799574,Benzofuran Library,Broad Institute of MIT and Harvard,NaN,NaN,COCC(=O)Nc1ccc2O[C@@H]3[C@@H](C[C@H](CC(=O)NCC...,NaN
30612,BRD-K30358777-001-01-0,BRD-K30358777,BROAD_CPD_ID,SA800605,Benzofuran Library,Broad Institute of MIT and Harvard,NaN,NaN,COc1ccc(CNC(=O)C[C@@H]2C[C@H]3[C@H](Oc4ccc(NC(...,NaN
30613,BRD-K32423836-001-01-9,BRD-K32423836,BROAD_CPD_ID,SA799806,Benzofuran Library,Broad Institute of MIT and Harvard,NaN,NaN,COCCNC(=O)C[C@H]1C[C@@H]2[C@@H](Oc3ccc(NC(=O)C...,NaN
30614,BRD-K28250273-001-01-2,BRD-K28250273,BROAD_CPD_ID,SA1482018,Azetidine Nitrile Full Library,Broad Institute of MIT and Harvard,NaN,NaN,OC[C@H]1[C@H]([C@H](C#N)N1C(=O)Nc1cccc(F)c1)c1...,NaN


In [83]:
# Function to convert SMILES to InChIKey
from rdkit import Chem
from rdkit.Chem import inchi
from pandarallel import pandarallel
pandarallel.initialize()

def smiles_to_inchikey(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles, sanitize=True)
        if mol is None:
            return 'CANNOT_CONVERT'
        # Optionally: kekulize or re-canonize
        Chem.SanitizeMol(mol)
        return inchi.MolToInchiKey(mol)
    except:
        return 'CANNOT_CONVERT'

moshkov= pd.read_csv("../../The_Moshkov_Files/data/assay_cp_data_PUMA.csv")
# Apply the function to the SMILES column

moshkov = moshkov.iloc[:, :271]

hof= pd.read_csv("../../The_Hofmarcher_Files/CP_count_Hofmarcher.csv")
hof = hof.iloc[:, :210]

ha = pd.read_csv("../../The_Ha_Files/CP_count_Ha.csv")
ha = ha.iloc[:, :202]

INFO: Pandarallel will run on 40 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [84]:
import pandas as pd

def filter_active_compounds(
    df,
    smiles_col,
    activity_value,
    min_hits,
    assay_cols,
    add_hit_count,
) -> pd.DataFrame:
    """
    Return a new DataFrame containing only compounds that are active
    in at least `min_hits` assays.

    Parameters
    ----------
    df : pd.DataFrame
        Input data containing a SMILES column plus assay‑result columns.
    smiles_col : str, default "smiles"
        Name of the SMILES identifier column.
    activity_value : int or float, default 1.0
        The value that denotes an active read‑out in the assay columns.
    min_hits : int, default 3
        Minimum number of assays in which a compound must be active to be kept.
    assay_cols : list[str] or None, default None
        List of assay columns. If None, all columns except `smiles_col` are used.
    add_hit_count : bool, default False
        If True, adds a column “n_active_hits” with the per‑compound hit count.

    Returns
    -------
    pd.DataFrame
        Filtered DataFrame of compounds meeting the activity criterion.
    """
    # Determine which columns to treat as assays
    if assay_cols is None:
        assay_cols = df.columns.drop(smiles_col)

    # Count active assays per row
    hit_counts = (df[assay_cols] == activity_value).sum(axis=1)

    # Filter rows
    keep_mask = hit_counts >= min_hits
    df_filtered = df.loc[keep_mask].copy()

    # Optionally append the hit count for reference
    if add_hit_count:
        df_filtered["n_active_hits"] = hit_counts[keep_mask]

    return df_filtered


# Assuming `df` is already loaded
moshkov = filter_active_compounds(moshkov, activity_value= 1.0, smiles_col="smiles",
                                    assay_cols = None,
                                    min_hits=2, add_hit_count=True)

hof = filter_active_compounds(hof, activity_value= 1.0, smiles_col="INCHIKEY",
                                    assay_cols = None,
                                    min_hits=2, add_hit_count=True)

ha = filter_active_compounds(ha, activity_value= 1.0, smiles_col="InChIKey",
                                    assay_cols = None,
                                    min_hits=2, add_hit_count=True)

print(f"Kept {len(df_active_moshkov)} compounds.")
print(f"Kept {len(df_active_hof)} compounds.")
print(f"Kept {len(df_active_ha)} compounds.")

Kept 3402 compounds.
Kept 4102 compounds.
Kept 4041 compounds.


In [85]:
moshkov['inchikey'] = moshkov['smiles'].parallel_apply(smiles_to_inchikey)

moshkov["key14"] = moshkov["inchikey"].str[:14]
hof["key14"] = hof["INCHIKEY"].str[:14]
ha["key14"] = ha["InChIKey"].str[:14]

moshkov = moshkov[["key14"]]
hof = hof[["key14"]]
ha = ha[["key14"]]

In [86]:
CPchem['inchikey'] = CPchem['CPD_SMILES'].parallel_apply(smiles_to_inchikey)
CPchem = CPchem[["inchikey", "BROAD_ID"]]
CPchem["key14"] = CPchem["inchikey"].str[:14]
CPchem

/opt/tljh/user/lib/python3.7/site-packages/ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  This is separate from the ipykernel package so we can avoid doing imports until


,inchikey,BROAD_ID,key14
0,VGLGVJVUHYTIIU-UHFFFAOYSA-N,BRD-A56675431-001-04-0,VGLGVJVUHYTIIU
1,QJHCTHPYUOXOGM-UHFFFAOYSA-N,BRD-A51829654-001-01-4,QJHCTHPYUOXOGM
2,WKRLQDKEXYKHJB-HFTRVMKXSA-N,BRD-K04046242-001-03-6,WKRLQDKEXYKHJB
3,AAOVKJBEBIDNHE-UHFFFAOYSA-N,BRD-K16508793-001-01-8,AAOVKJBEBIDNHE
4,WPVVMKYQOMJPIN-UHFFFAOYSA-N,BRD-K09397065-001-01-6,WPVVMKYQOMJPIN
...,...,...,...
30611,NDUFHUOGHUIKBH-OFYLTFBOSA-N,BRD-K47092271-001-01-7,NDUFHUOGHUIKBH
30612,UPMRBEOCYVNRMT-OXPRRSNCSA-N,BRD-K30358777-001-01-0,UPMRBEOCYVNRMT
30613,HRBYSLULMJVYNQ-TXDFXGTJSA-N,BRD-K32423836-001-01-9,HRBYSLULMJVYNQ
30614,NDKYDEPQZPJKDZ-ULQDDVLXSA-N,BRD-K28250273-001-01-2,NDKYDEPQZPJKDZ


In [89]:
# Gather overlaps from each source
overlap_frames = []
for name, df in zip(["moshkov", "hof", "ha"], [moshkov, hof, ha]):
    ov = (
        df[df["key14"].isin(CPchem["key14"])]      # rows that match master keys
          .merge(CPchem[["key14", "BROAD_ID"]],    # bring in BROAD_ID
                 on="key14", how="left")
    )                        
    overlap_frames.append(ov)

# Combine into a single DataFrame
all_overlaps = pd.concat(overlap_frames, ignore_index=True)

print(f"Total overlaps across all sources: {len(all_overlaps)}")
all_overlaps = all_overlaps.drop_duplicates(subset=["BROAD_ID"], keep="first").reset_index(drop=True)

Total overlaps across all sources: 28015


In [90]:
all_overlaps.to_csv("bioactive_compounds_with_assay_data.csv", index=False)